# LeetCode 2333 — Minimum Sum of Squared Difference bằng Python

Link đề: https://leetcode.com/problems/minimum-sum-of-squared-difference/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Greedy, Sorting, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def minSumSquareDiff(self, nums1: List[int], nums2: List[int], k1: int, k2: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Rút gọn bài: chỉ có **hiệu tuyệt đối** `dᵢ = |nums1[i] − nums2[i]|` quan trọng; mỗi thao tác (trên mảng nào cũng vậy) giảm một `dᵢ` đi 1 ⇒ gộp `k = k1 + k2`.
2. Tham lam **giảm hiệu lớn nhất** vì `d² − (d − 1)² = 2d − 1` tăng theo `d` (hàm lồi) — chứng minh bằng hoán đổi.
3. Không thể mô phỏng từng thao tác (`k` tới `2·10⁹`) ⇒ **san phẳng** (leveling): đưa mọi hiệu lớn về một ngưỡng `T` chung, rồi phân phối phần dư.
4. Tìm ngưỡng `T` bằng **đếm phân phối** (giá trị ≤ 10⁵), **sắp xếp**, hoặc **tìm nhị phân**.
5. Xử lý biên: `k ≥ Σ dᵢ` ⇒ đáp án 0 (không được "giảm quá" vì sẽ làm hiệu tăng trở lại).

## 2. Đề bài & ví dụ

Cho hai mảng dương `nums1`, `nums2` cùng độ dài `n`. Tổng bình phương hiệu là `Σ (nums1[i] − nums2[i])²`. Có thể tăng/giảm 1 phần tử của `nums1` tối đa `k1` lần, và của `nums2` tối đa `k2` lần (phần tử được phép âm). Trả về tổng bình phương hiệu **nhỏ nhất**.

**Ví dụ 1:** `nums1 = [1,2,3,4]`, `nums2 = [2,10,20,19]`, `k1 = k2 = 0` → `579`.

**Ví dụ 2:** `nums1 = [1,4,10,12]`, `nums2 = [5,8,6,9]`, `k1 = 1`, `k2 = 1` → `43`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; `0 ≤ nums[i] ≤ 10⁵`; `0 ≤ k1, k2 ≤ 10⁹`.

In [1]:
from typing import List
import heapq, random, time
from itertools import product

examples = [([1, 2, 3, 4], [2, 10, 20, 19], 0, 0, 579), ([1, 4, 10, 12], [5, 8, 6, 9], 1, 1, 43)]
for a, b, k1, k2, exp in examples:
    print(f"hiệu tuyệt đối {[abs(x - y) for x, y in zip(a, b)]}, k = {k1 + k2} -> kỳ vọng {exp}")

hiệu tuyệt đối [1, 8, 17, 15], k = 0 -> kỳ vọng 579
hiệu tuyệt đối [4, 4, 4, 3], k = 2 -> kỳ vọng 43


## 3. Kiến thức nền

### 3.1 Mọi thao tác chỉ tác động lên `dᵢ = |nums1[i] − nums2[i]|`

Tăng/giảm `nums1[i]` hoặc `nums2[i]` đi 1 làm hiệu `nums1[i] − nums2[i]` thay đổi đúng 1 ⇒ `|·|` thay đổi đúng 1 (giảm nếu đi về phía 0, tăng nếu ngược lại). Mảng nào bị sửa không quan trọng ⇒ tổng số thao tác `k = k1 + k2` dùng tự do cho mọi `i`.

### 3.2 Lợi ích biên giảm dần — hàm lồi

Giảm `d` đi 1 làm tổng giảm `d² − (d − 1)² = 2d − 1` — **lớn hơn** khi `d` lớn hơn. ⇒ Thao tác nên dồn vào hiệu lớn nhất.

In [2]:
for d in [1, 2, 3, 5, 10]:
    print(f"d = {d:>2}: giảm 1 làm tổng giảm {d * d - (d - 1) ** 2}")

d =  1: giảm 1 làm tổng giảm 1
d =  2: giảm 1 làm tổng giảm 3
d =  3: giảm 1 làm tổng giảm 5
d =  5: giảm 1 làm tổng giảm 9
d = 10: giảm 1 làm tổng giảm 19


### 3.3 San phẳng (leveling)

Với `k` thao tác, kết quả tham lam là: tồn tại ngưỡng `T` sao cho mọi `dᵢ > T` bị đưa về `T` (hoặc `T − 1` cho một số phần tử, do phần dư), còn `dᵢ ≤ T` giữ nguyên. Chi phí để đưa về `T`: `cost(T) = Σ max(0, dᵢ − T)` — **không tăng** theo `T`.

### 3.4 Không giảm quá 0

Nếu `k ≥ Σ dᵢ`, mọi hiệu về 0 ⇒ đáp án 0 (thao tác thừa không cần dùng — đề nói "tối đa").

## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: `d = [4, 4, 4, 3]`, `k = 2`. Tham lam: giảm một số 4 thành 3 (lợi 7), giảm một số 4 khác thành 3 (lợi 7) ⇒ `d = [3, 3, 4, 3]` ⇒ `9 + 9 + 16 + 9 = 43` ✔.

Bức tranh san phẳng: xếp các `dᵢ` như các cột, "đổ" `k` đơn vị từ đỉnh cột cao nhất xuống — giống mực nước rút: mọi cột cao hơn mức nước `T` bị cắt về `T`.

Với ví dụ 2: `cost(4) = 0`, `cost(3) = 3 > 2` ⇒ không đủ để đưa mọi cột về 3. Đưa về `T = 4` (cost 0), còn dư `r = 2` ⇒ 2 trong 3 cột có độ cao 4 giảm xuống 3.

In [3]:
def greedy_steps(d, k):
    h = [-x for x in d]; heapq.heapify(h); log = []
    for _ in range(k):
        x = -heapq.heappop(h)
        if x == 0: heapq.heappush(h, 0); break
        heapq.heappush(h, -(x - 1)); log.append(x)
    return sorted(-x for x in h), log

d = [abs(x - y) for x, y in zip(*examples[1][:2])]
final, log = greedy_steps(d, 2)
print("ban đầu", d, "| giảm lần lượt các cột cao", log, "| cuối", final, "| tổng", sum(x * x for x in final))

ban đầu [4, 4, 4, 3] | giảm lần lượt các cột cao [4, 4] | cuối [3, 3, 3, 4] | tổng 43


## 5. Cách 1 — Brute Force / baseline

### 5.1 Vét cạn mọi cách phân phối `k` thao tác lên `n` hiệu (n, k nhỏ)

Theo 3.1, chỉ cần chọn `tᵢ ≥ 0` với `Σ tᵢ ≤ k`, hiệu mới `max(0, dᵢ − tᵢ)`… — chính xác hơn `|dᵢ − tᵢ|` (giảm quá 0 thì hiệu tăng lại). Duyệt mọi bộ `tᵢ`.

In [4]:
def brute(nums1, nums2, k1, k2):
    d = [abs(x - y) for x, y in zip(nums1, nums2)]; k = k1 + k2; best = None
    for t in product(range(k + 1), repeat=len(d)):
        if sum(t) <= k:
            v = sum((di - ti) ** 2 for di, ti in zip(d, t))       # (d - t)^2 = |d - t|^2
            best = v if best is None else min(best, v)
    return best

for a, b, k1, k2, exp in examples:
    print(brute(a, b, k1, k2), "OK" if brute(a, b, k1, k2) == exp else "SAI")

579 OK
43 OK


Chỉ dùng được với `n ≤ 4`, `k ≤ 6`. Lưu ý `(d − t)²` đã tự xử lý việc "giảm quá 0".

**Kiểm chứng 3.1 (gộp `k1 + k2`)** bằng vét cạn trực tiếp trên hai mảng — thử mọi cách dùng `k1` thao tác trên `nums1` và `k2` trên `nums2` (mỗi thao tác ±1 trên một phần tử):

In [5]:
def brute_two_arrays(nums1, nums2, k1, k2):
    n = len(nums1)
    def adjust(arr, k):                               # mọi mảng đạt được với tối đa k thao tác ±1
        res = set()
        for delta in product(range(-k, k + 1), repeat=n):
            if sum(map(abs, delta)) <= k: res.add(tuple(x + y for x, y in zip(arr, delta)))
        return res
    return min(sum((x - y) ** 2 for x, y in zip(A, B)) for A in adjust(nums1, k1) for B in adjust(nums2, k2))

rng = random.Random(5)
for _ in range(60):
    n = rng.randint(1, 3); a = [rng.randint(0, 6) for _ in range(n)]; b = [rng.randint(0, 6) for _ in range(n)]
    k1, k2 = rng.randint(0, 2), rng.randint(0, 2)
    assert brute_two_arrays(a, b, k1, k2) == brute(a, b, k1, k2)
print("gộp k = k1 + k2 khớp vét cạn trên hai mảng")

gộp k = k1 + k2 khớp vét cạn trên hai mảng


### 5.2 Baseline 2 — Heap: mỗi thao tác giảm hiệu lớn nhất — O(k log n)

Đúng (mục 10), nhưng `k` tới `2·10⁹` ⇒ quá chậm.

In [6]:
def heap_greedy(nums1, nums2, k1, k2):
    d = [abs(x - y) for x, y in zip(nums1, nums2)]; k = k1 + k2
    if sum(d) <= k: return 0
    h = [-x for x in d]; heapq.heapify(h)
    for _ in range(k):
        x = -heapq.heappop(h); heapq.heappush(h, -(x - 1))
    return sum(x * x for x in h)

for a, b, k1, k2, exp in examples:
    print(heap_greedy(a, b, k1, k2), "OK" if heap_greedy(a, b, k1, k2) == exp else "SAI")

579 OK
43 OK


**Nút thắt:** các thao tác lặp lại "giảm cùng mức" rất nhiều lần — gộp chúng lại theo **mức** (bucket) hoặc tìm thẳng ngưỡng `T`.

## 6. Ý tưởng thuật toán / bước đột phá

**Đếm phân phối theo mức (cách doocs).** `cnt[v]` = số hiệu bằng `v`. Duyệt `v` từ lớn xuống: chuyển `take = min(cnt[v], k)` phần tử từ mức `v` xuống mức `v − 1` (mỗi phần tử tốn 1 thao tác). Mỗi mức tốn O(1) ⇒ O(n + V).

**Tìm ngưỡng trực tiếp.** Tìm `T` nhỏ nhất với `cost(T) = Σ max(0, dᵢ − T) ≤ k` (tìm nhị phân, hoặc quét trên mảng sắp xếp). Đưa mọi `dᵢ > T` về `T`; còn dư `r = k − cost(T)` thao tác — `r` nhỏ hơn số phần tử đang ở mức `T` (vì nếu không, `T − 1` cũng khả thi), nên giảm đúng `r` phần tử từ `T` xuống `T − 1`.

Đáp án `= Σ_{dᵢ ≤ T, không bị chạm} dᵢ² + (c − r)·T² + r·(T − 1)²` với `c` = số phần tử ở mức `T` sau khi san phẳng.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đếm phân phối theo mức (cách doocs)

In [7]:
def bucket(nums1, nums2, k1, k2):
    k = k1 + k2; cnt = [0] * 100001; s = mx = 0
    for a, b in zip(nums1, nums2):
        v = abs(a - b); cnt[v] += 1; s += v; mx = max(mx, v)
    if s <= k: return 0
    for v in range(mx, 0, -1):
        if cnt[v] == 0: continue
        take = min(cnt[v], k)
        k -= take; cnt[v] -= take; cnt[v - 1] += take
        if k == 0: break
    return sum(v * v * c for v, c in enumerate(cnt) if c)

for a, b, k1, k2, exp in examples:
    print(bucket(a, b, k1, k2), "OK" if bucket(a, b, k1, k2) == exp else "SAI")

579 OK
43 OK


Bản nộp ở mục 9 chỉ cấp phát `cnt` tới `max d + 1` thay vì cố định `100001` — kết quả như nhau nhưng nhanh hơn nhiều khi gọi hàm hàng nghìn lần trong bộ test (cấp phát và duyệt `10⁵` ô mỗi lần là đáng kể).

Đi xuống từng mức: tại mức `v`, các phần tử đang ở `v` (gồm cả những phần tử vừa được đẩy xuống từ `v + 1`) được đẩy xuống `v − 1`. Nếu không đủ thao tác, chỉ đẩy `k` phần tử rồi dừng — đó chính là "phần dư" của cách san phẳng.

### 7.2 Cách 3 — Tìm nhị phân ngưỡng `T`

In [8]:
def binary_search(nums1, nums2, k1, k2):
    d = [abs(a - b) for a, b in zip(nums1, nums2)]; k = k1 + k2
    if sum(d) <= k: return 0
    cost = lambda T: sum(x - T for x in d if x > T)
    lo, hi = 0, max(d)                                  # cost(max) = 0 <= k
    while lo < hi:
        mid = (lo + hi) // 2
        if cost(mid) <= k: hi = mid
        else: lo = mid + 1
    T = lo; r = k - cost(T)
    new = [min(x, T) for x in d]
    at_T = sum(1 for x in new if x == T)
    assert T > 0 and r < at_T
    return sum(x * x for x in new) - r * (T * T - (T - 1) * (T - 1))

for a, b, k1, k2, exp in examples:
    print(binary_search(a, b, k1, k2), "OK" if binary_search(a, b, k1, k2) == exp else "SAI")

579 OK
43 OK


`T > 0` vì `Σ d > k` nghĩa là `cost(0) > k`. `r < at_T`: nếu `r ≥ at_T` thì đưa mọi phần tử ở `T` xuống `T − 1` cũng đủ, tức `cost(T − 1) ≤ k`, mâu thuẫn với `T` nhỏ nhất. O(n log V).

### 7.3 Cách 4 — Sắp xếp giảm dần và "rút mực nước" theo từng bậc — O(n log n)

In [9]:
def sorted_leveling(nums1, nums2, k1, k2):
    d = sorted((abs(a - b) for a, b in zip(nums1, nums2)), reverse=True); k = k1 + k2; n = len(d)
    if sum(d) <= k: return 0
    d.append(0)
    i = 0                                              # d[0..i] đang cùng mức d[i]
    while True:
        while i + 1 < n and d[i + 1] == d[i]: i += 1
        nxt = d[i + 1]                                 # mức kế tiếp (0 nếu hết)
        need = (i + 1) * (d[i] - nxt)                  # đưa i+1 cột đầu xuống mức nxt
        if need <= k and i + 1 < n:
            k -= need
            for t in range(i + 1): d[t] = nxt
            i += 1
        else:
            q, r = divmod(k, i + 1)                    # hạ thêm q cho cả nhóm, r cột hạ thêm 1
            level = d[i] - q
            top = [level - 1] * r + [level] * (i + 1 - r)
            return sum(x * x for x in top) + sum(x * x for x in d[i + 1:n])

for a, b, k1, k2, exp in examples:
    print(sorted_leveling(a, b, k1, k2), "OK" if sorted_leveling(a, b, k1, k2) == exp else "SAI")

579 OK
43 OK


Vòng lặp nhóm dần các cột cao nhất (có cùng mức), hạ cả nhóm xuống mức của cột kế tiếp nếu đủ thao tác; nếu không đủ thì chia đều `k` còn lại cho nhóm (`divmod`). Cài đặt trên đơn giản hóa bằng cách gán lại `d[t]` — O(n²) worst; bản O(n log n) chỉ cần lưu mức chung của nhóm thay vì gán từng phần tử (cách 5).

### 7.4 Cách 5 — Sắp xếp + mức chung của nhóm (O(n log n) thật sự)

In [10]:
def sorted_group(nums1, nums2, k1, k2):
    d = sorted((abs(a - b) for a, b in zip(nums1, nums2)), reverse=True); k = k1 + k2; n = len(d)
    if sum(d) <= k: return 0
    level = d[0]; g = 1                               # nhóm g cột đầu đang ở cùng mức `level`
    while True:
        while g < n and d[g] == level: g += 1
        nxt = d[g] if g < n else 0
        need = g * (level - nxt)
        if need <= k and g < n:
            k -= need; level = nxt
        else:
            q, r = divmod(k, g); level -= q
            return r * (level - 1) ** 2 + (g - r) * level ** 2 + sum(x * x for x in d[g:])

for a, b, k1, k2, exp in examples:
    print(sorted_group(a, b, k1, k2), "OK" if sorted_group(a, b, k1, k2) == exp else "SAI")

579 OK
43 OK


## 8. TRACE từng bước

### 8.1 Trace cách đếm phân phối trên một ví dụ nhiều mức

In [11]:
def trace_bucket(d, k):
    cnt = {}
    for v in d: cnt[v] = cnt.get(v, 0) + 1
    print("ban đầu:", dict(sorted(cnt.items(), reverse=True)), "k =", k)
    for v in range(max(d), 0, -1):
        c = cnt.get(v, 0)
        if not c: continue
        take = min(c, k); k -= take
        cnt[v] = c - take; cnt[v - 1] = cnt.get(v - 1, 0) + take
        print(f"  mức {v}: đẩy {take} phần tử xuống {v - 1} -> {dict(sorted(((a, b) for a, b in cnt.items() if b), reverse=True))}, k còn {k}")
        if k == 0: break
    return sum(v * v * c for v, c in cnt.items())

print("đáp án:", trace_bucket([9, 7, 7, 2], 6))

ban đầu: {9: 1, 7: 2, 2: 1} k = 6
  mức 9: đẩy 1 phần tử xuống 8 -> {8: 1, 7: 2, 2: 1}, k còn 5
  mức 8: đẩy 1 phần tử xuống 7 -> {7: 3, 2: 1}, k còn 4
  mức 7: đẩy 3 phần tử xuống 6 -> {6: 3, 2: 1}, k còn 1
  mức 6: đẩy 1 phần tử xuống 5 -> {6: 2, 5: 1, 2: 1}, k còn 0
đáp án: 101


`[9, 7, 7, 2]`, `k = 6`: mức 9 → 8 (1 thao tác), mức 8 → 7 (1), giờ có ba số 7: đẩy cả ba xuống 6 (3 thao tác), còn 1: đẩy một số 6 xuống 5 ⇒ `[6, 6, 5, 2]` ⇒ `36 + 36 + 25 + 4 = 101`.

### 8.2 Trace tìm nhị phân ngưỡng

In [12]:
d = [9, 7, 7, 2]; k = 6
for T in range(9, 3, -1):
    print(f"T = {T}: cost = {sum(x - T for x in d if x > T)}", "<= k" if sum(x - T for x in d if x > T) <= k else "> k")
print("T nhỏ nhất khả thi = 6 (cost 5), dư r = 1 -> một phần tử 6 xuống 5")

T = 9: cost = 0 <= k
T = 8: cost = 1 <= k
T = 7: cost = 2 <= k
T = 6: cost = 5 <= k
T = 5: cost = 8 > k
T = 4: cost = 11 > k
T nhỏ nhất khả thi = 6 (cost 5), dư r = 1 -> một phần tử 6 xuống 5


## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def minSumSquareDiff(self, nums1: List[int], nums2: List[int], k1: int, k2: int) -> int:
        k = k1 + k2                                   # thao tác trên mảng nào cũng giảm |hiệu| đi 1
        diffs = [abs(a - b) for a, b in zip(nums1, nums2)]
        total, top = sum(diffs), max(diffs)
        if total <= k:
            return 0                                  # đưa được mọi hiệu về 0
        cnt = [0] * (top + 1)                         # chỉ cần tới mức cao nhất
        for v in diffs:
            cnt[v] += 1
        for v in range(top, 0, -1):                   # hạ dần từ mức cao nhất (hàm lồi => tham lam)
            if cnt[v]:
                move = min(cnt[v], k)
                k -= move
                cnt[v] -= move
                cnt[v - 1] += move
                if k == 0:
                    break
        return sum(v * v * c for v, c in enumerate(cnt) if c)

In [14]:
sol = Solution()
for a, b, k1, k2, exp in examples:
    got = sol.minSumSquareDiff(a, b, k1, k2)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

579 | kỳ vọng 579 | OK
43 | kỳ vọng 43 | OK


## 10. Vì sao thuật toán đúng

**Bước 1 — rút gọn.** Theo 3.1, bài tương đương: cho `dᵢ ≥ 0`, chọn `tᵢ ≥ 0` nguyên với `Σ tᵢ ≤ k` để tối thiểu `Σ (dᵢ − tᵢ)²`. (Thao tác "tăng hiệu" không bao giờ có ích; mọi phân bổ giữa `nums1` và `nums2` đều thực hiện được vì cả hai cùng tác động lên `dᵢ`.)

**Bước 2 — không giảm quá 0.** Nếu `tᵢ > dᵢ`, thay `tᵢ` bằng `dᵢ` không làm tăng số thao tác và không làm tăng `(dᵢ − tᵢ)²` ⇒ giả sử `tᵢ ≤ dᵢ`. Nếu `k ≥ Σ dᵢ`, chọn `tᵢ = dᵢ` cho 0.

**Bước 3 — tham lam đúng (hoán đổi).** Gọi `eᵢ = dᵢ − tᵢ` (giá trị cuối). Trong nghiệm tối ưu với `Σ tᵢ = k` (dùng hết là tốt nhất khi `k < Σ d`), không tồn tại `i, j` với `eᵢ ≥ eⱼ + 2` mà `tⱼ > 0`: nếu có, chuyển một thao tác từ `j` sang `i` (`eᵢ − 1`, `eⱼ + 1`) làm tổng thay đổi `(eᵢ − 1)² + (eⱼ + 1)² − eᵢ² − eⱼ² = 2(eⱼ − eᵢ) + 2 ≤ −2 < 0` — mâu thuẫn. Nghĩa là: mọi phần tử bị chạm (`tⱼ > 0`) có giá trị cuối nằm trong `{L − 1, L}` với `L = max eᵢ`, và mọi phần tử không bị chạm có `dᵢ ≤ L`. Đây đúng là cấu hình "san phẳng" với ngưỡng `T = L`; cấu hình này được xác định duy nhất (đa tập giá trị cuối) bởi `k` ⇒ mọi cách tính san phẳng (đếm phân phối, tìm nhị phân, sắp xếp) cho cùng tổng tối ưu. ∎

**Đếm phân phối mô phỏng đúng san phẳng.** Duyệt mức từ cao xuống, tại mỗi mức đẩy càng nhiều phần tử càng tốt xuống mức dưới — tương đương với áp dụng tham lam "giảm phần tử lớn nhất" theo lô; dừng khi hết thao tác.

## 11. Phân tích Time / Space Complexity

`V = max dᵢ ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn phân phối | O((k+1)ⁿ · n) | O(n) |
| 1b. Heap từng thao tác | O(k log n) | O(n) |
| 2. Đếm phân phối (doocs) | O(n + V) | O(V) (bản nộp: O(max d)) |
| 3. Tìm nhị phân `T` | O(n log V) | O(n) |
| 4. Sắp xếp + gán lại | O(n log n + n²) worst | O(n) |
| 5. Sắp xếp + mức chung | O(n log n) | O(n) |

In [15]:
rng = random.Random(2333)
A = [rng.randint(0, 10**5) for _ in range(10**5)]; B = [rng.randint(0, 10**5) for _ in range(10**5)]
S = sum(abs(a - b) for a, b in zip(A, B))
for k1, k2 in [(0, 0), (10**6, 10**6), (S // 3, S // 3), (10**9, 10**9)]:
    res = []
    for name, f in [("bucket", bucket), ("binary_search", binary_search), ("sorted_group", sorted_group),
                    ("Solution", Solution().minSumSquareDiff)]:
        t0 = time.perf_counter(); res.append(f(A, B, k1, k2)); dt = time.perf_counter() - t0
        print(f"k={k1 + k2:<12} {name:<14} {dt*1000:7.1f} ms -> {res[-1]}")
    assert len(set(res)) == 1
t0 = time.perf_counter(); heap_greedy(A[:2000], B[:2000], 10**5, 10**5); print(f"heap_greedy (n=2000, k=2e5): {(time.perf_counter()-t0)*1000:.0f} ms — k = 2e9 sẽ gấp 10^4 lần")

k=0            bucket            15.3 ms -> 167421160691877


k=0            binary_search     48.6 ms -> 167421160691877


k=0            sorted_group      27.1 ms -> 167421160691877
k=0            Solution          13.0 ms -> 167421160691877


k=2000000      bucket            14.0 ms -> 167046963546793


k=2000000      binary_search     40.6 ms -> 167046963546793
k=2000000      sorted_group      23.9 ms -> 167046963546793
k=2000000      Solution          13.6 ms -> 167046963546793
k=2227548386   bucket            24.9 ms -> 13479802108699


k=2227548386   binary_search     79.5 ms -> 13479802108699


k=2227548386   sorted_group      36.0 ms -> 13479802108699


k=2227548386   Solution          24.2 ms -> 13479802108699


k=2000000000   bucket            26.9 ms -> 19940206661405
k=2000000000   binary_search     84.7 ms -> 19940206661405
k=2000000000   sorted_group      34.0 ms -> 19940206661405


k=2000000000   Solution          23.0 ms -> 19940206661405


heap_greedy (n=2000, k=2e5): 46 ms — k = 2e9 sẽ gấp 10^4 lần


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [16]:
ALL = {"brute": brute, "heap_greedy": heap_greedy, "bucket": bucket, "binary_search": binary_search,
       "sorted_leveling": sorted_leveling, "sorted_group": sorted_group, "Solution": Solution().minSumSquareDiff}
edge = examples + [
    ([5], [5], 3, 0, 0),                  # đã bằng nhau
    ([0], [10], 3, 4, 9),                 # 10 - 7 = 3
    ([0], [10], 10, 0, 0),                # vừa đủ
    ([0], [10], 6, 6, 0),                 # thừa thao tác: không được giảm quá 0
    ([0, 0], [3, 3], 3, 0, 5),            # [3,3] -> [2,1]? cân bằng: [2,1] = 5 hoặc [1,2]
    ([0, 0, 0], [5, 1, 1], 2, 0, 11),     # [5,1,1] -> [3,1,1]
]
for a, b, k1, k2, exp in edge:
    for name, f in ALL.items():
        if name == "brute" and k1 + k2 > 7: continue
        got = f(a, b, k1, k2)
        assert got == exp, (name, a, b, k1, k2, got, exp)
    print(f"d={[abs(x - y) for x, y in zip(a, b)]!s:<10} k={k1 + k2:<3} -> {exp}")
print("edge OK")

d=[1, 8, 17, 15] k=0   -> 579
d=[4, 4, 4, 3] k=2   -> 43
d=[0]        k=3   -> 0
d=[10]       k=7   -> 9
d=[10]       k=10  -> 0
d=[10]       k=12  -> 0
d=[3, 3]     k=3   -> 5
d=[5, 1, 1]  k=2   -> 11
edge OK


### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

In [17]:
rng = random.Random(42)
for _ in range(500):
    n = rng.randint(1, 4); a = [rng.randint(0, 9) for _ in range(n)]; b = [rng.randint(0, 9) for _ in range(n)]
    k1, k2 = rng.randint(0, 3), rng.randint(0, 3)
    ref = brute(a, b, k1, k2)
    for name, f in ALL.items():
        assert f(a, b, k1, k2) == ref, (name, a, b, k1, k2)
print("500 bộ ngẫu nhiên OK")

500 bộ ngẫu nhiên OK


### 12.3 Tính chất: không tăng khi `k` tăng; khớp heap với `n`, `k` vừa; chỉ phụ thuộc `k1 + k2`

In [18]:
rng = random.Random(7)
for _ in range(200):
    n = rng.randint(1, 30); a = [rng.randint(0, 50) for _ in range(n)]; b = [rng.randint(0, 50) for _ in range(n)]
    vals = [Solution().minSumSquareDiff(a, b, k, 0) for k in range(0, 60, 3)]
    assert all(x >= y for x, y in zip(vals, vals[1:]))
    k = rng.randint(0, 300)
    assert Solution().minSumSquareDiff(a, b, k, 0) == heap_greedy(a, b, k, 0) == Solution().minSumSquareDiff(a, b, k // 2, k - k // 2)
print("tính chất OK")

tính chất OK


### 12.4 Edge cực trị: `n = 10⁵` hiệu đều bằng `10⁵`, `k1 = k2 = 10⁹`; đáp án vượt 32 bit

In [19]:
n = 10**5
A = [0] * n; B = [10**5] * n
S = n * 10**5                                                  # tổng hiệu = 10^10
print("k = 0:", Solution().minSumSquareDiff(A, B, 0, 0), "| vượt int32:", n * 10**10 > 2**31 - 1)
assert Solution().minSumSquareDiff(A, B, 0, 0) == n * 10**10
k = 2 * 10**9
print("tổng hiệu", S, "> k =", k, "=> không về 0 được; mỗi hiệu giảm đúng k / n =", k // n)
v = Solution().minSumSquareDiff(A, B, 10**9, 10**9)
assert v == n * (10**5 - k // n) ** 2 == binary_search(A, B, 10**9, 10**9)
print("đáp án:", v, "= n * (10^5 - 2*10^4)^2")

k = 0: 1000000000000000 | vượt int32: True
tổng hiệu 10000000000 > k = 2000000000 => không về 0 được; mỗi hiệu giảm đúng k / n = 20000
đáp án: 640000000000000 = n * (10^5 - 2*10^4)^2


Mọi hiệu bằng nhau ⇒ san phẳng chia đều: mỗi hiệu giảm `2·10⁹ / 10⁵ = 2·10⁴`, còn `8·10⁴` ⇒ đáp án `10⁵ · (8·10⁴)² = 6,4·10¹⁴` — cần số nguyên 64 bit ở C++/Java. Đây cũng là trường hợp nặng nhất cho cách đếm phân phối (phải đi qua `2·10⁴` mức với `10⁵` phần tử mỗi mức — vẫn chỉ O(V) bước vì mỗi mức là O(1)).

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Giảm **quá 0** (không kiểm tra `Σ d ≤ k`), làm hiệu âm rồi bình phương lại tăng

In [20]:
def wrong_overshoot(nums1, nums2, k1, k2):
    d = [abs(a - b) for a, b in zip(nums1, nums2)]; k = k1 + k2
    q, r = divmod(k, len(d))                                    # SAI: chia đều mọi thao tác cho mọi phần tử
    return sum((x - q - (1 if i < r else 0)) ** 2 for i, x in enumerate(sorted(d, reverse=True)))

for a, b, k1, k2 in [([0], [10], 6, 6), ([0, 0], [1, 10], 4, 0), (examples[1][0], examples[1][1], 1, 1)]:
    print("bản sai :", wrong_overshoot(a, b, k1, k2), "| bản đúng:", Solution().minSumSquareDiff(a, b, k1, k2), "cho", a, b, k1, k2)

bản sai : 4 | bản đúng: 0 cho [0] [10] 6 6
bản sai : 65 | bản đúng: 37 cho [0, 0] [1, 10] 4 0
bản sai : 43 | bản đúng: 43 cho [1, 4, 10, 12] [5, 8, 6, 9] 1 1


**Vì sao sai:** chia đều bỏ qua việc các hiệu nhỏ không cần (và không nên) giảm, và còn giảm quá 0. `[0], [10]`, `k = 12`: `10 − 12 = −2` ⇒ 4 thay vì 0. Ví dụ 2 **trùng hợp** đúng (`d = [4, 4, 4, 3]` gần đều).

### Lỗi 2 — Xử lý `k1` và `k2` **riêng** với hai lượt tham lam khác nhau trên hai mảng mà quên cập nhật hiệu

In [21]:
def wrong_separate(nums1, nums2, k1, k2):
    d = [abs(a - b) for a, b in zip(nums1, nums2)]
    i = max(range(len(d)), key=d.__getitem__)
    d1 = d[:]; d1[i] = max(0, d1[i] - k1)                        # SAI: dồn hết k1 vào MỘT phần tử
    j = max(range(len(d1)), key=d1.__getitem__)
    d1[j] = max(0, d1[j] - k2)                                   # rồi dồn hết k2 vào một phần tử khác
    return sum(x * x for x in d1)

for a, b, k1, k2, _ in examples + [([0, 0], [10, 9], 3, 0, None), ([0, 0, 0], [6, 6, 6], 2, 2, None)]:
    print("bản sai :", wrong_separate(a, b, k1, k2), "| bản đúng:", Solution().minSumSquareDiff(a, b, k1, k2), "cho d =", [abs(x - y) for x, y in zip(a, b)], "k1, k2 =", k1, k2)

bản sai : 579 | bản đúng: 579 cho d = [1, 8, 17, 15] k1, k2 = 0 0
bản sai : 43 | bản đúng: 43 cho d = [4, 4, 4, 3] k1, k2 = 1 1
bản sai : 130 | bản đúng: 128 cho d = [10, 9] k1, k2 = 3 0
bản sai : 68 | bản đúng: 66 cho d = [6, 6, 6] k1, k2 = 2 2


**Vì sao sai:** hai ví dụ đề **trùng hợp** đúng (`k1, k2 ≤ 1`: dồn một thao tác vào phần tử lớn nhất chính là tham lam đúng). `d = [10, 9]`, `k1 = 3`: bản sai cho `[7, 9]` (49 + 81 = 130), tối ưu là `[8, 8]` (128). `d = [6, 6, 6]`, `k1 = k2 = 2`: bản sai dồn 2 thao tác vào một số 6 rồi 2 thao tác vào số 6 khác ⇒ `[4, 4, 6]` (68); tối ưu chia 4 thao tác cho cả ba số ⇒ `[5, 5, 4]` (25 + 25 + 16 = 66). Dồn nhiều thao tác vào một phần tử bỏ lỡ lợi ích lớn hơn ở các phần tử khác sau khi phần tử đó đã nhỏ đi — phải chọn lại phần tử lớn nhất **sau mỗi** thao tác (hoặc san phẳng).

### Lỗi 3 — Dùng heap **nhỏ nhất** (giảm hiệu nhỏ nhất trước)

In [22]:
def wrong_min_heap(nums1, nums2, k1, k2):
    d = [abs(a - b) for a, b in zip(nums1, nums2)]; k = k1 + k2
    h = [x for x in d if x > 0]; heapq.heapify(h); rest = [x for x in d if x == 0]
    for _ in range(k):
        if not h: break
        x = heapq.heappop(h) - 1                                 # SAI: lợi ích nhỏ nhất
        (heapq.heappush(h, x) if x > 0 else rest.append(0))
    return sum(x * x for x in h)

for a, b, k1, k2, _ in examples:
    print("bản sai :", wrong_min_heap(a, b, k1, k2), "| bản đúng:", Solution().minSumSquareDiff(a, b, k1, k2))

bản sai : 579 | bản đúng: 579
bản sai : 49 | bản đúng: 43


**Vì sao sai:** giảm `d` nhỏ chỉ lợi `2d − 1` nhỏ. Ví dụ 2: giảm hai lần ở số 3 (`3 → 1`) lợi `5 + 3 = 8`, trong khi giảm hai số 4 lợi `7 + 7 = 14`. Ví dụ 1 (`k = 0`) **trùng hợp** đúng.

### Lỗi 4 — Tìm nhị phân `T` nhưng **quên phần dư** `r`

In [23]:
def wrong_no_remainder(nums1, nums2, k1, k2):
    d = [abs(a - b) for a, b in zip(nums1, nums2)]; k = k1 + k2
    if sum(d) <= k: return 0
    cost = lambda T: sum(x - T for x in d if x > T)
    lo, hi = 0, max(d)
    while lo < hi:
        mid = (lo + hi) // 2
        if cost(mid) <= k: hi = mid
        else: lo = mid + 1
    return sum(min(x, lo) ** 2 for x in d)                       # SAI: bỏ phí k - cost(T) thao tác

for a, b, k1, k2, _ in examples:
    print("bản sai :", wrong_no_remainder(a, b, k1, k2), "| bản đúng:", Solution().minSumSquareDiff(a, b, k1, k2))

bản sai : 579 | bản đúng: 579
bản sai : 57 | bản đúng: 43


**Vì sao sai:** ví dụ 2: `T = 4` (cost 0), phần dư 2 thao tác bị bỏ phí ⇒ 57 thay vì 43. Ví dụ 1 (`k = 0`) **trùng hợp** đúng.

### Lỗi 5 — Không gộp `k1 + k2` mà chỉ dùng `k1` (hiểu nhầm "`k2` chỉ cho `nums2` nên không giúp được")

In [24]:
def wrong_only_k1(nums1, nums2, k1, k2):
    return Solution().minSumSquareDiff(nums1, nums2, k1, 0)      # SAI: bỏ k2

for a, b, k1, k2, _ in examples:
    print("bản sai :", wrong_only_k1(a, b, k1, k2), "| bản đúng:", Solution().minSumSquareDiff(a, b, k1, k2))

bản sai : 579 | bản đúng: 579
bản sai : 50 | bản đúng: 43


**Sửa:** sửa `nums2[i]` cũng thay đổi `|nums1[i] − nums2[i]|` đúng 1 — hai loại thao tác tương đương.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Mục tiêu **lồi** (tổng bình phương, …) + "giảm 1 đơn vị mỗi thao tác" ⇒ tham lam **giảm phần tử lớn nhất**.
- `k` rất lớn ⇒ không mô phỏng từng bước; **san phẳng**: tìm ngưỡng `T` (đếm phân phối / tìm nhị phân / sắp xếp) rồi chia phần dư.
- Hai nguồn thao tác cùng tác động lên một đại lượng ⇒ **gộp** lại.
- Luôn xử lý biên "đủ để về 0".
- Bài cùng họ: 1648 Sell Diminishing-Valued Colored Balls, 2233 Maximum Product After K Increments, 1642 Furthest Building You Can Reach, 2448 Minimum Cost to Make Array Equal, 1508, 2208 Minimum Operations to Halve Array Sum.

## 15. Biến thể của bài

### 15.1 Biến thể A — Trọng số theo chỉ số: tối thiểu `Σ wᵢ · dᵢ²` (vẫn lồi ⇒ heap theo lợi ích biên)

In [25]:
def weighted(d, w, k):
    h = [(-(wi * (2 * di - 1)), i) for i, (di, wi) in enumerate(zip(d, w)) if di > 0]; heapq.heapify(h)
    d = d[:]
    for _ in range(k):
        if not h: break
        _, i = heapq.heappop(h); d[i] -= 1
        if d[i] > 0: heapq.heappush(h, (-(w[i] * (2 * d[i] - 1)), i))
    return sum(wi * di * di for di, wi in zip(d, w))

def weighted_brute(d, w, k):
    return min(sum(wi * (di - ti) ** 2 for di, wi, ti in zip(d, w, t))
               for t in product(range(k + 1), repeat=len(d)) if sum(t) <= k and all(ti <= di for ti, di in zip(t, d)))

rng = random.Random(151)
for _ in range(300):
    n = rng.randint(1, 3); d = [rng.randint(0, 6) for _ in range(n)]; w = [rng.randint(1, 5) for _ in range(n)]; k = rng.randint(0, 6)
    assert weighted(d, w, k) == weighted_brute(d, w, k)
print(weighted([4, 4, 4, 3], [1, 1, 1, 1], 2), weighted([4, 4, 4, 3], [1, 1, 5, 1], 2))

43 61


### 15.2 Biến thể B — Tổng **trị tuyệt đối** `Σ dᵢ` (tuyến tính): phân phối thế nào cũng như nhau

In [26]:
def min_abs_sum(nums1, nums2, k1, k2):
    return max(0, sum(abs(a - b) for a, b in zip(nums1, nums2)) - (k1 + k2))
print(min_abs_sum(*examples[0][:4]), min_abs_sum(*examples[1][:4]))

41 13


Hàm tuyến tính không có lợi ích biên giảm dần ⇒ mọi thao tác (miễn là không giảm quá 0) đều lợi đúng 1.

### 15.3 Biến thể C — Mỗi phần tử chỉ được giảm tối đa `capᵢ` lần

In [27]:
def capped(d, cap, k):
    h = [(-(2 * di - 1), i) for i, di in enumerate(d) if di > 0 and cap[i] > 0]; heapq.heapify(h)
    d = d[:]; used = [0] * len(d)
    for _ in range(k):
        if not h: break
        _, i = heapq.heappop(h); d[i] -= 1; used[i] += 1
        if d[i] > 0 and used[i] < cap[i]: heapq.heappush(h, (-(2 * d[i] - 1), i))
    return sum(x * x for x in d)

def capped_brute(d, cap, k):
    return min(sum((di - ti) ** 2 for di, ti in zip(d, t))
               for t in product(*[range(min(c, di) + 1) for c, di in zip(cap, d)]) if sum(t) <= k)

rng = random.Random(153)
for _ in range(300):
    n = rng.randint(1, 4); d = [rng.randint(0, 6) for _ in range(n)]; cap = [rng.randint(0, 4) for _ in range(n)]; k = rng.randint(0, 8)
    assert capped(d, cap, k) == capped_brute(d, cap, k)
print(capped([9, 7, 7, 2], [1, 5, 5, 5], 6))

109


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | định nghĩa | mũ | O(n) | kiểm chứng |
| 1b. Heap từng bước | giảm max mỗi lần | O(k log n) | O(n) | `k` nhỏ / biến thể có trọng số |
| 2. Đếm phân phối | hạ theo mức | O(n + V) | O(V) | **mặc định** (V ≤ 10⁵) |
| 3. Tìm nhị phân `T` | ngưỡng + phần dư | O(n log V) | O(n) | V lớn |
| 5. Sắp xếp + mức chung | rút mực nước | O(n log n) | O(n) | V lớn, không muốn log V |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minSumSquareDiff([1, 1], [4, 1], 1, 1)` và `minSumSquareDiff([0, 0], [2, 2], 1, 2)`?

**Bài 2 (tính tay):** `d = [10, 8, 8, 3]`, `k = 7`. Tính ngưỡng `T`, phần dư và đáp án.

**Bài 3 (điền TODO):** Viết `bt3_todo(d, k)` trả về **mảng hiệu cuối** (đã sắp giảm dần) theo san phẳng.

**Bài 4 (tự viết lại):** Viết `bt4_todo(nums1, nums2, k1, k2)` cài cách **tìm nhị phân ngưỡng** không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(nums1, nums2, k1, k2)` trả về **số thao tác ít nhất** cần dùng để đạt đáp án tối ưu (có thể nhỏ hơn `k1 + k2`).

In [28]:
def bt3_todo(d, k):
    # TODO: mảng hiệu cuối
    pass

def bt4_todo(nums1, nums2, k1, k2):
    # TODO: tìm nhị phân ngưỡng
    pass

def bt5_todo(nums1, nums2, k1, k2):
    # TODO: số thao tác ít nhất đạt tối ưu
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `d = [3, 0]`, `k = 2`. Mức 2: `3 → 1`. Mức 3: `1`; câu hai: `d = [2, 2]`, `k = 3` ⇒ `[1, 0]` ⇒ `1`.

**Bài 2.** Mức 1: `cost(8) = 2`, `cost(7) = 1 + 1 + 3 = 5`, `cost(6) = 4 + 2 + 2 = 8 > 7` ⇒ `T = 7`. Mức 2: dư `r = 2`, có 3 phần tử ở mức 7. Mức 3: `[7, 6, 6, 3]` ⇒ `49 + 36 + 36 + 9 = 130`.

**Bài 3.** Mức 1: tìm `T`, `r`. Mức 2: `min(d, T)` cho mọi phần tử. Mức 3: hạ `r` phần tử đang ở `T` xuống `T − 1`.

**Bài 4.** Mức 1: `cost(T)`. Mức 2: tìm `T` nhỏ nhất `cost(T) ≤ k`. Mức 3: trừ `r · (2T − 1)`.

**Bài 5.** Mức 1: nếu `Σ d ≤ k` thì cần đúng `Σ d`. Mức 2: ngược lại mọi thao tác đều có lợi (mỗi bước giảm `2d − 1 ≥ 1` khi `d ≥ 1`). Mức 3: ⇒ `min(Σ d, k1 + k2)`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `1`, `1`. **Bài 2:** `T = 7`, `r = 2`, mảng `[7, 6, 6, 3]`, đáp án `130`.

In [29]:
assert Solution().minSumSquareDiff([1, 1], [4, 1], 1, 1) == 1
assert Solution().minSumSquareDiff([0, 0], [2, 2], 1, 2) == 1
assert Solution().minSumSquareDiff([0, 0, 0, 0], [10, 8, 8, 3], 7, 0) == 130

def bt3(d, k):
    if sum(d) <= k: return [0] * len(d)
    cost = lambda T: sum(x - T for x in d if x > T)
    lo, hi = 0, max(d)
    while lo < hi:
        mid = (lo + hi) // 2
        if cost(mid) <= k: hi = mid
        else: lo = mid + 1
    r = k - cost(lo); new = sorted((min(x, lo) for x in d), reverse=True)
    for i in range(len(new)):
        if r and new[i] == lo: new[i] -= 1; r -= 1
    return sorted(new, reverse=True)

def bt4(nums1, nums2, k1, k2):
    return binary_search(nums1, nums2, k1, k2)

def bt5(nums1, nums2, k1, k2):
    return min(sum(abs(a - b) for a, b in zip(nums1, nums2)), k1 + k2)

rng = random.Random(2003)
for _ in range(400):
    n = rng.randint(1, 4); a = [rng.randint(0, 9) for _ in range(n)]; b = [rng.randint(0, 9) for _ in range(n)]
    k1, k2 = rng.randint(0, 3), rng.randint(0, 3)
    d = [abs(x - y) for x, y in zip(a, b)]
    assert sum(x * x for x in bt3(d, k1 + k2)) == brute(a, b, k1, k2) and sum(d) - sum(bt3(d, k1 + k2)) == min(sum(d), k1 + k2)
    assert bt4(a, b, k1, k2) == brute(a, b, k1, k2)
    best = brute(a, b, k1, k2); used = bt5(a, b, k1, k2)
    assert brute(a, b, used, 0) == best and (used == 0 or brute(a, b, used - 1, 0) > best)
print(bt3([10, 8, 8, 3], 7), bt5(*examples[1][:4]))
print("Đáp án bài tập: tất cả assert OK")

[7, 6, 6, 3] 2
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
k = k1 + k2
cnt = [0] * (10**5 + 1); total = top = 0
for a, b in zip(nums1, nums2):
    v = abs(a - b); cnt[v] += 1; total += v; top = max(top, v)
if total <= k: return 0
for v in range(top, 0, -1):            # hạ từ mức cao nhất
    if cnt[v]:
        move = min(cnt[v], k); k -= move
        cnt[v] -= move; cnt[v - 1] += move
        if k == 0: break
return sum(v * v * c for v, c in enumerate(cnt))
```

**Checklist:**
- [ ] Gộp `k = k1 + k2`.
- [ ] `Σ d ≤ k` ⇒ 0 (không giảm quá 0).
- [ ] Giảm phần tử **lớn nhất** (lồi), theo lô — không mô phỏng từng thao tác.
- [ ] Nhớ phần dư `r` khi dùng ngưỡng `T`.